# RAG Exercise 2 — Build a RAG System Over Your Own Documents

This builds the full pipeline end-to-end: **load documents -> chunk -> embed & retrieve -> generate an answer from retrieved context -> wrap it in a LangChain chain -> evaluate.**

It reuses the same tools as Exercise 1 (`SentenceTransformer('all-MiniLM-L6-v2')`, a HuggingFace generation `pipeline`, and LangChain) — just applied to real files in a `my_docs/` folder.

## 1. Collect & Prepare Your Own Documents

A `my_docs/` folder sits next to this notebook with **3 text documents** I chose so the retrieval examples are meaningful:

| File | Topic | Why |
|---|---|---|
| `ai_ethics.md` | Ethical challenges in AI (bias, transparency, privacy, accountability) | Lets us ask about *AI ethics* |
| `ai_sustainability.txt` | AI for climate & sustainability | Different but related domain, tests retrieval precision |
| `transformers_and_rag.txt` | Transformers, embeddings, and how RAG works | So the system can explain its own machinery |

> Swap in your own `.txt` / `.md` files anytime — the code just loads everything in the folder.

In [ ]:
import os

folder = 'my_docs'
documents = []
filenames = []

for file in sorted(os.listdir(folder)):
    if file.endswith(('.txt', '.md')):
        with open(os.path.join(folder, file), 'r', encoding='utf-8') as f:
            documents.append(f.read())
            filenames.append(file)

print(f'Loaded {len(documents)} documents: {filenames}')
print('\n--- preview of first document ---')
print(documents[0][:300])

## 2. Chunk Your Texts

Long documents are split into smaller **chunks** so retrieval can return just the relevant piece instead of a whole file. Below we also try chunk sizes 100 / 200 / 400 words to see the trade-off.

In [ ]:
def chunk_text(text, chunk_size=200):
    words = text.split()
    return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

# Experiment: how many chunks does each size produce?
for size in (100, 200, 400):
    total = sum(len(chunk_text(doc, size)) for doc in documents)
    print(f'chunk_size={size:>3}  ->  {total} chunks')

# We use 150 as a balance: small enough to be focused, big enough to keep context.
CHUNK_SIZE = 150
chunks = []
for doc in documents:
    chunks.extend(chunk_text(doc, CHUNK_SIZE))

print(f'\nUsing chunk_size={CHUNK_SIZE} -> {len(chunks)} total chunks')

**Which chunk size works best?** Very small chunks (100) retrieve a precise sentence but can lose surrounding context; very large chunks (400) keep context but dilute relevance and risk overflowing the generator's input limit. For these short documents **~150 words** gave the most relevant retrievals while staying within `distilgpt2`'s small context window.

## 3. Build Your Own Retriever (Semantic)

Embed every chunk once, then for a query embed it and rank chunks by **cosine similarity**.

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embedder.encode(chunks, convert_to_tensor=True)

def retrieve_chunks(query, k=2):
    query_embed = embedder.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_embed, chunk_embeddings)[0]
    top_k = torch.topk(scores, k)
    return [chunks[i] for i in top_k.indices]

# TEST
for c in retrieve_chunks('What is discussed about AI ethics?', k=2):
    print('-', c[:120], '...\n')

## 4. Connect Retrieval with Generation

Retrieve chunks, paste them into a prompt as **context**, and let the model answer.

In [ ]:
from transformers import pipeline

generator = pipeline('text-generation', model='distilgpt2', max_new_tokens=120, temperature=0.3)

def mini_rag(query, k=2):
    retrieved = retrieve_chunks(query, k)
    context = '\n'.join(retrieved)
    prompt = f'Use the following context to answer the question:\n\n{context}\n\nQuestion: {query}\n\nAnswer:'
    return generator(prompt)[0]['generated_text']

for q in ['Summarize the main idea about AI ethics.',
          'How does AI help with sustainability?',
          'What is retrieval-augmented generation?']:
    print('Q:', q)
    print(mini_rag(q), '\n' + '='*80)

### Same question **with vs without** retrieval

This is the core point of RAG: does giving the model retrieved context change the answer?

In [ ]:
question = 'What are the main challenges in AI ethics?'

print('--- WITHOUT retrieval (model alone) ---')
print(generator(f'Question: {question}\nAnswer:')[0]['generated_text'])

print('\n--- WITH retrieval (RAG) ---')
print(mini_rag(question, k=2))

**Observation.** Without retrieval, `distilgpt2` (a tiny, non-instruction model) tends to ramble or invent text — it has no grounding. With retrieval, the prompt contains the actual passage about bias, transparency, privacy and accountability, so the answer stays closer to the source. `distilgpt2` is still weak; in section 6 we swap in `flan-t5-base`, which uses the context far better.

## 5. Add a LangChain RAG Chain

Same logic, expressed as a modular LangChain pipeline: `{context, question} | prompt | llm`. Per the TODO we retrieve **top 3** chunks and **print them** before answering. (Imports use the current `langchain_core` / `langchain_community` paths.)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_community.llms import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=generator)

prompt = ChatPromptTemplate.from_template(
    'Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer clearly and concisely:'
)

def get_context(query, k=3):
    retrieved = retrieve_chunks(query, k)
    print(f'--- Retrieved top {k} chunks ---')
    for i, c in enumerate(retrieved, 1):
        print(f'[{i}] {c[:100]}...')
    print('-'*60)
    return '\n'.join(retrieved)

rag_chain = (
    {'context': get_context, 'question': RunnablePassthrough()}
    | prompt
    | llm
)

print(rag_chain.invoke('What are the key challenges mentioned about AI?'))

**Did more chunks help?** Going from `top_k=2` to `top_k=3` adds context but also adds noise and uses more of `distilgpt2`'s tiny input window, so quality can actually drop. For a stronger model (`flan-t5-base`) the extra chunk usually helps because it can use the longer context.

## 6. Home Assignment — Try Another Model (`flan-t5-base`)

`flan-t5-base` is **instruction-tuned** and seq2seq, so it follows the 'answer using the context' instruction much better than `distilgpt2`. We run the same question through both to compare.

In [ ]:
flan = pipeline('text2text-generation', model='google/flan-t5-base', max_new_tokens=120)

def rag_with(model_pipe, query, k=3, text_key='generated_text'):
    context = '\n'.join(retrieve_chunks(query, k))
    prompt = f'Answer the question using only the context.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:'
    return model_pipe(prompt)[0][text_key]

q = 'What are the main challenges in AI ethics?'
print('=== distilgpt2 ===')
print(rag_with(generator, q))
print('\n=== flan-t5-base ===')
print(rag_with(flan, q))

**Comparison.** `flan-t5-base` produces a short, on-topic answer that actually reflects the retrieved context (bias, transparency, privacy, accountability), while `distilgpt2` tends to continue the text loosely. Same retrieval, very different generation quality — the generator matters as much as the retriever.

## 7. Home Assignment — Visualize Cosine Similarity

Bar chart of the similarity score between the query and each **retrieved** chunk, to see how confident the ranking is.

In [ ]:
import matplotlib.pyplot as plt

def plot_similarity(query, k=5):
    query_embed = embedder.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_embed, chunk_embeddings)[0]
    top = torch.topk(scores, min(k, len(chunks)))
    labels = [chunks[i][:35] + '...' for i in top.indices]
    values = [float(v) for v in top.values]

    plt.figure(figsize=(8, 4))
    plt.barh(labels[::-1], values[::-1], color='teal')
    plt.xlabel('Cosine similarity')
    plt.title(f'Top {k} chunks for: "{query}"')
    plt.tight_layout()
    plt.show()

plot_similarity('What are the main challenges in AI ethics?', k=5)

## 8. Evaluation & Reflection

| **Question** | **Answer** |
|---|---|
| **How many documents did you use?** | 3 text documents in `my_docs/` (AI ethics, AI sustainability, transformers/RAG). |
| **What type of content (topic/domain)?** | Short explanatory articles on AI topics — ethics, climate/sustainability, and NLP/RAG fundamentals. |
| **Which retrieval size (chunk length, `top_k`) worked best?** | `chunk_size ~150` with `top_k=2-3`. Smaller chunks improved precision; `top_k=3` helped flan-t5 but slightly hurt distilgpt2 (too much text for its small window). |
| **Did the model produce hallucinations? When?** | Yes — `distilgpt2` hallucinated/rambled, especially **without retrieval** and when context was long. `flan-t5-base` stayed grounded far more often. |
| **What improvement would you try next?** | Use a stronger instruction model (e.g. flan-t5-large or a hosted Mistral), add sentence-overlap chunking, and store embeddings in a vector DB (FAISS/Chroma) for scale. |

### Summary

Retrieval grounds the generator in real text and clearly reduces hallucination compared to the model answering alone. But retrieval quality and generator quality are **both** needed: good chunks fed to a weak model (`distilgpt2`) still give weak answers, while the same chunks fed to `flan-t5-base` give coherent, on-topic responses.